In [1]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine

# 1. Configurar la conexión a PostgreSQL
usuario = 'postgres'
contrasena = '14678261' # Reemplaza esto con tu contraseña real (deja las comillas)
host = 'localhost'
puerto = '5432'
base_datos = 'postgres'

# Crear la cadena de conexión para SQLAlchemy
conexion_url = f'postgresql+psycopg2://{usuario}:{contrasena}@{host}:{puerto}/{base_datos}'
motor = create_engine(conexion_url)

# 2. Consultar las tablas unidas para simular un reporte logístico y de ventas
query = """
    SELECT 
        v.id_venta,
        c.nombre_cliente,
        c.ciudad,
        p.nombre AS producto,
        p.categoria,
        v.cantidad,
        p.precio,
        p.costo,
        (v.cantidad * p.precio) AS ingreso_total,
        (v.cantidad * (p.precio - p.costo)) AS ganancia_total,
        v.fecha_venta,
        v.estado_entrega
    FROM ventas v
    JOIN clientes c ON v.id_cliente = c.id_cliente
    JOIN productos p ON v.id_producto = p.id_producto;
"""

# 3. Cargar los datos en un DataFrame de Pandas
df = pd.read_sql(query, motor)

# Mostrar las primeras filas del DataFrame para verificar que todo cargó bien
print("¡Datos cargados exitosamente desde PostgreSQL!\n")
display(df.head())

¡Datos cargados exitosamente desde PostgreSQL!



,id_venta,nombre_cliente,ciudad,producto,categoria,cantidad,precio,costo,ingreso_total,ganancia_total,fecha_venta,estado_entrega
0,1,Supermercado El Ahorro,Cali,Café Molido 500g,Alimentos,150,18000.0,12000.0,2700000.0,900000.0,2026-09-01,Entregado
1,2,Minimarket La 14,Bogotá,Galletas de Avena,Snacks,300,6000.0,3500.0,1800000.0,750000.0,2026-09-03,Entregado
2,3,Distribuciones del Valle,Cali,Agua Botellada 600ml,Bebidas,500,2500.0,1200.0,1250000.0,650000.0,2026-09-05,Retrasado
3,4,Supermercado El Ahorro,Cali,Chocolate Oscuro,Snacks,80,12000.0,7000.0,960000.0,400000.0,2026-09-10,Entregado
4,5,Comercializadora Andina,Medellín,Café Molido 500g,Alimentos,200,18000.0,12000.0,3600000.0,1200000.0,2026-09-12,En tránsito


In [2]:
# 1. Resumen general de ganancias totales e ingresos totales
total_ingresos = df['ingreso_total'].sum()
total_ganancias = df['ganancia_total'].sum()

print(f"--- REPORTE FINANCIERO LOGÍSTICO ---")
print(f"Ingresos Totales: ${total_ingresos:,.2f}")
print(f"Ganancias Totales: ${total_ganancias:,.2f}\n")

# 2. Análisis del estado de las entregas (logística)
print("--- ESTADO DE LAS ENTREGAS ---")
entregas_estado = df['estado_entrega'].value_counts()
print(entregas_estado)


--- REPORTE FINANCIERO LOGÍSTICO ---
Ingresos Totales: $11,310,000.00
Ganancias Totales: $4,420,000.00

--- ESTADO DE LAS ENTREGAS ---
estado_entrega
Entregado      4
Retrasado      1
En tránsito    1
Name: count, dtype: int64


In [3]:
# 3. Agrupar por categoría de producto para ver cuáles generan más ingresos
ventas_por_categoria = df.groupby('categoria')[['ingreso_total', 'ganancia_total']].sum().reset_index()

print("--- RENDIMIENTO POR CATEGORÍA ---")
display(ventas_por_categoria)


--- RENDIMIENTO POR CATEGORÍA ---


,categoria,ingreso_total,ganancia_total
0,Alimentos,6300000.0,2100000.0
1,Bebidas,2250000.0,1170000.0
2,Snacks,2760000.0,1150000.0
